# prepare

In [29]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import re
import warnings
from pathlib import Path
warnings.filterwarnings("ignore")

from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

BASE_PATH = "../../../../../data/longds/education/bi/task1"

import builtins as _builtins
_ORIGINAL_ROUND = _builtins.round
_ORIGINAL_JSON_DUMPS = json.dumps
_ORIGINAL_DF_ROUND = pd.DataFrame.round
_ORIGINAL_SERIES_ROUND = pd.Series.round

def round(value, ndigits=None):
    if ndigits is None:
        return _ORIGINAL_ROUND(value)
    return value

def _round_noop(self, decimals=0, *args, **kwargs):
    return self.copy()

pd.DataFrame.round = _round_noop
pd.Series.round = _round_noop

def _round_for_json(value):
    if isinstance(value, dict):
        return {k: _round_for_json(v) for k, v in value.items()}
    if isinstance(value, list):
        return [_round_for_json(v) for v in value]
    if isinstance(value, tuple):
        return [_round_for_json(v) for v in value]
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float):
        if np.isnan(value) or np.isinf(value):
            return value
        return _ORIGINAL_ROUND(float(value), 4)
    return value

def json_dumps_rounded(value, *args, **kwargs):
    return _ORIGINAL_JSON_DUMPS(_round_for_json(value), *args, **kwargs)


# Task 1

## Context

This dataset contains records of BI (Business Intelligence) program students.
Throughout this analysis, compute derived quantities such as sums, gaps, ratios, means, or correlations using unrounded values, and report all decimal-valued final results rounded to 4 decimal places

## Q

Load the student dataset and report the total number of rows and columns, the name and data type of each column, and the number of missing values per column. How many total missing values are there across the entire dataset?

## C

In [30]:
df = pd.read_csv(f"{BASE_PATH}/data/intro-to-data-cleaning-eda-and-machine-learning/bi.csv", encoding='latin1')

shape_info = {"rows": int(df.shape[0]), "columns": int(df.shape[1])}
dtypes_info = {col: str(dtype) for col, dtype in df.dtypes.items()}
nulls_info = {col: int(v) for col, v in df.isnull().sum().items()}

result = {
    "answer_type": "dataset_overview",
    "shape": shape_info,
    "column_dtypes": dtypes_info,
    "missing_values": nulls_info,
    "total_missing": int(df.isnull().sum().sum())
}
print(json_dumps_rounded(result, ensure_ascii=False, indent=2))

{
  "answer_type": "dataset_overview",
  "shape": {
    "rows": 77,
    "columns": 11
  },
  "column_dtypes": {
    "fNAME": "str",
    "lNAME": "str",
    "Age": "int64",
    "gender": "str",
    "country": "str",
    "residence": "str",
    "entryEXAM": "int64",
    "prevEducation": "str",
    "studyHOURS": "int64",
    "Python": "float64",
    "DB": "int64"
  },
  "missing_values": {
    "fNAME": 0,
    "lNAME": 0,
    "Age": 0,
    "gender": 0,
    "country": 0,
    "residence": 0,
    "entryEXAM": 0,
    "prevEducation": 0,
    "studyHOURS": 0,
    "Python": 2,
    "DB": 0
  },
  "total_missing": 2
}


# Task 2

## Context

The dataset has missing values in the Python score column. These should be imputed with the column mean so that the overall distribution shape is preserved. We also need to verify there are no duplicate student records before proceeding with further analysis.

## Q

Fill all missing values in the Python score column with the column mean. Then check whether any duplicate rows exist. Report the mean value used for imputation, the number of duplicates found, and confirm the total missing values remaining across all columns.

## C

In [31]:
python_mean = float(df['Python'].mean())
df['Python'] = df['Python'].fillna(python_mean)
num_duplicates = int(df.duplicated().sum())

result = {
    "answer_type": "scalar",
    "python_fill_mean": round(python_mean, 4),
    "num_duplicates": num_duplicates,
    "missing_values_after": {col: int(v) for col, v in df.isnull().sum().items()},
    "total_missing_after": int(df.isnull().sum().sum())
}
print(json_dumps_rounded(result, ensure_ascii=False, indent=2))

{
  "answer_type": "scalar",
  "python_fill_mean": 75.8533,
  "num_duplicates": 0,
  "missing_values_after": {
    "fNAME": 0,
    "lNAME": 0,
    "Age": 0,
    "gender": 0,
    "country": 0,
    "residence": 0,
    "entryEXAM": 0,
    "prevEducation": 0,
    "studyHOURS": 0,
    "Python": 0,
    "DB": 0
  },
  "total_missing_after": 0
}


# Task 3

## Q

Calculate the summary statistics (count, mean, standard deviation, min, 25th percentile, median, 75th percentile, max) for all numeric columns in the dataset after imputation. Report the full statistical summary for each numeric column.

## C

In [32]:
desc = df.describe()
stats = {}
for col in desc.columns:
    stats[col] = {k: round(float(v), 4) for k, v in desc[col].items()}

result = {"answer_type": "distribution_summary", "statistics": stats}
print(json_dumps_rounded(result, ensure_ascii=False, indent=2))

{
  "answer_type": "distribution_summary",
  "statistics": {
    "Age": {
      "count": 77.0,
      "mean": 35.2078,
      "std": 10.342,
      "min": 21.0,
      "25%": 27.0,
      "50%": 33.0,
      "75%": 42.0,
      "max": 71.0
    },
    "entryEXAM": {
      "count": 77.0,
      "mean": 76.7532,
      "std": 16.4758,
      "min": 28.0,
      "25%": 69.0,
      "50%": 80.0,
      "75%": 90.0,
      "max": 98.0
    },
    "studyHOURS": {
      "count": 77.0,
      "mean": 149.7143,
      "std": 12.7433,
      "min": 114.0,
      "25%": 144.0,
      "50%": 156.0,
      "75%": 158.0,
      "max": 160.0
    },
    "Python": {
      "count": 77.0,
      "mean": 75.8533,
      "std": 15.2062,
      "min": 15.0,
      "25%": 72.0,
      "50%": 81.0,
      "75%": 85.0,
      "max": 91.0
    },
    "DB": {
      "count": 77.0,
      "mean": 69.4675,
      "std": 17.0337,
      "min": 30.0,
      "25%": 56.0,
      "50%": 71.0,
      "75%": 83.0,
      "max": 100.0
    }
  }
}


# Task 4

## Context

## Q

Standardize all column names to PascalCase. Report the complete mapping from each original column name to its cleaned version.

## C

In [33]:
original_cols = df.columns.tolist()

def clean_column(name):
    words = re.findall(r'[A-Z]?[a-z]+|[A-Z]+(?![a-z])', name)
    return ''.join([w.capitalize() for w in words])

df.columns = [clean_column(col) for col in df.columns]
cleaned_cols = df.columns.tolist()

mapping = [{"original": o, "cleaned": c} for o, c in zip(original_cols, cleaned_cols)]
print(json_dumps_rounded({"answer_type": "column_mapping", "mapping": mapping}, ensure_ascii=False, indent=2))

{
  "answer_type": "column_mapping",
  "mapping": [
    {
      "original": "fNAME",
      "cleaned": "FName"
    },
    {
      "original": "lNAME",
      "cleaned": "LName"
    },
    {
      "original": "Age",
      "cleaned": "Age"
    },
    {
      "original": "gender",
      "cleaned": "Gender"
    },
    {
      "original": "country",
      "cleaned": "Country"
    },
    {
      "original": "residence",
      "cleaned": "Residence"
    },
    {
      "original": "entryEXAM",
      "cleaned": "EntryExam"
    },
    {
      "original": "prevEducation",
      "cleaned": "PrevEducation"
    },
    {
      "original": "studyHOURS",
      "cleaned": "StudyHours"
    },
    {
      "original": "Python",
      "cleaned": "Python"
    },
    {
      "original": "DB",
      "cleaned": "Db"
    }
  ]
}


# Task 5

## Q

Standardize the country names Report the number of unique countries before and after cleaning, and the student count per country after cleaning sorted in descending order. Which are the top 4 countries by student count?

## C

In [34]:
unique_before = int(df['Country'].nunique())

country_mapping = {
    'Rsa': 'South Africa',
    'Norge': 'Norway',
    'norway': 'Norway',
    'UK': 'United Kingdom',
    'Somali': 'Somalia'
}
df['Country'] = df['Country'].replace(country_mapping)

counts = df['Country'].value_counts().reset_index()
counts.columns = ['country', 'student_count']

result = {
    "answer_type": "ranking",
    "unique_before": unique_before,
    "unique_after": int(df['Country'].nunique()),
    "top4": counts.head(4).to_dict(orient='records'),
    "all_countries": counts.to_dict(orient='records')
}
print(json_dumps_rounded(result, ensure_ascii=False, indent=2))

{
  "answer_type": "ranking",
  "unique_before": 16,
  "unique_after": 13,
  "top4": [
    {
      "country": "Norway",
      "student_count": 49
    },
    {
      "country": "Uganda",
      "student_count": 4
    },
    {
      "country": "Kenya",
      "student_count": 3
    },
    {
      "country": "Germany",
      "student_count": 3
    }
  ],
  "all_countries": [
    {
      "country": "Norway",
      "student_count": 49
    },
    {
      "country": "Uganda",
      "student_count": 4
    },
    {
      "country": "Kenya",
      "student_count": 3
    },
    {
      "country": "Germany",
      "student_count": 3
    },
    {
      "country": "South Africa",
      "student_count": 2
    },
    {
      "country": "Denmark",
      "student_count": 2
    },
    {
      "country": "Netherlands",
      "student_count": 2
    },
    {
      "country": "Italy",
      "student_count": 2
    },
    {
      "country": "Spain",
      "student_count": 2
    },
    {
      "country": "United 

# Task 6

## Q

Standardize the education level values. Report the number of unique levels before and after cleaning, and the student count per education level after cleaning, sorted in descending order.

## C

In [35]:
unique_before = int(df['PrevEducation'].nunique())

education_mapping = {
    'HighSchool': 'High School',
    'Barrrchelors': 'Bachelors',
    'diploma': 'Diploma',
    'DIPLOMA': 'Diploma',
    'Diplomaaa': 'Diploma'
}
df['PrevEducation'] = df['PrevEducation'].replace(education_mapping)

counts = df['PrevEducation'].value_counts().reset_index()
counts.columns = ['education_level', 'student_count']

result = {
    "answer_type": "ranking",
    "unique_before": unique_before,
    "unique_after": int(df['PrevEducation'].nunique()),
    "education_counts": counts.to_dict(orient='records')
}
print(json_dumps_rounded(result, ensure_ascii=False, indent=2))

{
  "answer_type": "ranking",
  "unique_before": 10,
  "unique_after": 5,
  "education_counts": [
    {
      "education_level": "Bachelors",
      "student_count": 25
    },
    {
      "education_level": "High School",
      "student_count": 19
    },
    {
      "education_level": "Masters",
      "student_count": 16
    },
    {
      "education_level": "Diploma",
      "student_count": 12
    },
    {
      "education_level": "Doctorate",
      "student_count": 5
    }
  ]
}


# Task 7

## Context

## Q

Standardize all gender values. Report the count of students for each gender after cleaning.

## C

In [36]:
gender_mapping = {
    'F': 'Female',
    'female': 'Female',
    'M': 'Male',
    'male': 'Male'
}
df['Gender'] = df['Gender'].replace(gender_mapping)

counts = df['Gender'].value_counts().reset_index()
counts.columns = ['gender', 'student_count']

result = {
    "answer_type": "group_stat",
    "gender_counts": counts.to_dict(orient='records')
}
print(json_dumps_rounded(result, ensure_ascii=False, indent=2))

{
  "answer_type": "group_stat",
  "gender_counts": [
    {
      "gender": "Female",
      "student_count": 43
    },
    {
      "gender": "Male",
      "student_count": 34
    }
  ]
}


# Task 8

## Context

## Q

Standardize all residence values. Report the student count for each residence type after cleaning, sorted in descending order.

## C

In [37]:
residence_mapping = {
    'BI-Residence': 'BI Residence',
    'BIResidence': 'BI Residence',
    'BI_Residence': 'BI Residence'
}
df['Residence'] = df['Residence'].replace(residence_mapping)

counts = df['Residence'].value_counts().reset_index()
counts.columns = ['residence', 'student_count']

result = {
    "answer_type": "group_stat",
    "residence_counts": counts.to_dict(orient='records')
}
print(json_dumps_rounded(result, ensure_ascii=False, indent=2))

{
  "answer_type": "group_stat",
  "residence_counts": [
    {
      "residence": "Private",
      "student_count": 33
    },
    {
      "residence": "BI Residence",
      "student_count": 32
    },
    {
      "residence": "Sognsvann",
      "student_count": 12
    }
  ]
}


# Task 9

## Context

## Q

Combine the first name and last name into a single full name column named `Name`, place it as the first column in the dataset, and remove the original separate name columns. Report the first 5 full names created and the complete list of column names in the final dataset.

## C

In [38]:
df.insert(0, 'Name', df['FName'] + ' ' + df['LName'])
df = df.drop(columns=['FName', 'LName'])

result = {
    "answer_type": "scalar",
    "first_5_names": df['Name'].head(5).tolist(),
    "final_columns": df.columns.tolist(),
    "total_columns": len(df.columns)
}
print(json_dumps_rounded(result, ensure_ascii=False, indent=2))

{
  "answer_type": "scalar",
  "first_5_names": [
    "Christina Binger",
    "Alex Walekhwa",
    "Philip Leo",
    "Shoni Hlongwane",
    "Maria Kedibone"
  ],
  "final_columns": [
    "Name",
    "Age",
    "Gender",
    "Country",
    "Residence",
    "EntryExam",
    "PrevEducation",
    "StudyHours",
    "Python",
    "Db"
  ],
  "total_columns": 10
}


# Task 10

## Context

Outliers in the Age column should be detected using the Interquartile Range (IQR) method.

## Q

Apply the IQR method to detect outliers in student ages. Report Q1, Q3, IQR, the lower and upper bounds, the number of outliers found, and the outlier values with corresponding student names. Then replace the outliers with the column mean and report the age distribution statistics (mean, median, population standard deviation, min, max, Q1, Q3, IQR) after replacement.

## C

In [39]:
Q1_age = float(df['Age'].quantile(0.25))
Q3_age = float(df['Age'].quantile(0.75))
IQR_age = Q3_age - Q1_age
lower_bound = Q1_age - 1.5 * IQR_age
upper_bound = Q3_age + 1.5 * IQR_age

outlier_mask = (df['Age'] < lower_bound) | (df['Age'] > upper_bound)
outliers_df = df[outlier_mask][['Name', 'Age']].copy()
outlier_records = []
for _, row in outliers_df.iterrows():
    outlier_records.append({"name": str(row['Name']), "age": int(row['Age'])})

df['Age'] = df['Age'].astype(float)
mean_age = float(df['Age'].mean())
df.loc[outlier_mask, 'Age'] = mean_age

s = df['Age']
result = {
    "answer_type": "outlier_detection",
    "Q1": round(Q1_age, 4),
    "Q3": round(Q3_age, 4),
    "IQR": round(IQR_age, 4),
    "lower_bound": round(lower_bound, 4),
    "upper_bound": round(upper_bound, 4),
    "num_outliers": int(outlier_mask.sum()),
    "outliers": outlier_records,
    "mean_used_for_replacement": round(mean_age, 4),
    "post_replacement_stats": {
        "mean": round(float(s.mean()), 4),
        "median": round(float(s.median()), 4),
        "std": round(float(s.std(ddof=0)), 4),
        "min": round(float(s.min()), 4),
        "max": round(float(s.max()), 4),
        "q25": round(float(s.quantile(0.25)), 4),
        "q75": round(float(s.quantile(0.75)), 4),
        "iqr": round(float(s.quantile(0.75) - s.quantile(0.25)), 4)
    }
}
print(json_dumps_rounded(result, ensure_ascii=False, indent=2))

{
  "answer_type": "outlier_detection",
  "Q1": 27.0,
  "Q3": 42.0,
  "IQR": 15.0,
  "lower_bound": 4.5,
  "upper_bound": 64.5,
  "num_outliers": 2,
  "outliers": [
    {
      "name": "Perry Rønning",
      "age": 71
    },
    {
      "name": "Chinedu Okafor",
      "age": 69
    }
  ],
  "mean_used_for_replacement": 35.2078,
  "post_replacement_stats": {
    "mean": 34.3041,
    "median": 33.0,
    "std": 8.5606,
    "min": 21.0,
    "max": 60.0,
    "q25": 27.0,
    "q75": 41.0,
    "iqr": 14.0
  }
}


# Task 11

## Q

Calculate the distribution statistics for the four academic performance variables: study hours, entry exam score, Python course score, and database course score. For each variable, report the mean, median, population standard deviation, min, max, 25th percentile, 75th percentile, and interquartile range.

## C

In [40]:
result = {"answer_type": "distribution_summary", "metrics": {}}
for col in ['StudyHours', 'EntryExam', 'Python', 'Db']:
    s = df[col].dropna()
    result["metrics"][col] = {
        "mean": round(float(s.mean()), 4),
        "median": round(float(s.median()), 4),
        "std": round(float(s.std(ddof=0)), 4),
        "min": round(float(s.min()), 4),
        "max": round(float(s.max()), 4),
        "q25": round(float(s.quantile(0.25)), 4),
        "q75": round(float(s.quantile(0.75)), 4),
        "iqr": round(float(s.quantile(0.75) - s.quantile(0.25)), 4)
    }
print(json_dumps_rounded(result, ensure_ascii=False, indent=2))

{
  "answer_type": "distribution_summary",
  "metrics": {
    "StudyHours": {
      "mean": 149.7143,
      "median": 156.0,
      "std": 12.6603,
      "min": 114.0,
      "max": 160.0,
      "q25": 144.0,
      "q75": 158.0,
      "iqr": 14.0
    },
    "EntryExam": {
      "mean": 76.7532,
      "median": 80.0,
      "std": 16.3684,
      "min": 28.0,
      "max": 98.0,
      "q25": 69.0,
      "q75": 90.0,
      "iqr": 21.0
    },
    "Python": {
      "mean": 75.8533,
      "median": 81.0,
      "std": 15.1071,
      "min": 15.0,
      "max": 91.0,
      "q25": 72.0,
      "q75": 85.0,
      "iqr": 13.0
    },
    "Db": {
      "mean": 69.4675,
      "median": 71.0,
      "std": 16.9227,
      "min": 30.0,
      "max": 100.0,
      "q25": 56.0,
      "q75": 83.0,
      "iqr": 27.0
    }
  }
}


# Task 12

## Q

For each country with at least 5 students, calculate the average age and the average entry exam score. Which country has the highest average age and which has the highest average entry exam score? Report all qualifying countries with both metrics, sorted by average entry exam score in descending order.

## C

In [41]:
# --- re-derive: country student counts (from country standardization) ---
country_counts = df['Country'].value_counts()
qualifying_countries = country_counts[country_counts >= 5].index.tolist()

# --- re-derive: age is already outlier-treated in df (from outlier replacement) ---

# --- novel computation: country-level averages for qualifying countries ---
subset = df[df['Country'].isin(qualifying_countries)]
country_stats = (subset.groupby('Country')
                 .agg(avg_age=('Age', 'mean'),
                      avg_entry_exam=('EntryExam', 'mean'),
                      student_count=('Name', 'count'))
                 .round(4)
                 .sort_values('avg_entry_exam', ascending=False)
                 .reset_index())

highest_age_row = country_stats.loc[country_stats['avg_age'].idxmax()]
highest_exam_row = country_stats.iloc[0]

result = {
    "answer_type": "group_stat",
    "qualifying_countries": country_stats.to_dict(orient='records'),
    "highest_avg_age": {"country": str(highest_age_row['Country']),
                        "avg_age": round(float(highest_age_row['avg_age']), 4)},
    "highest_avg_entry_exam": {"country": str(highest_exam_row['Country']),
                               "avg_entry_exam": round(float(highest_exam_row['avg_entry_exam']), 4)}
}
print(json_dumps_rounded(result, ensure_ascii=False, indent=2))

{
  "answer_type": "group_stat",
  "qualifying_countries": [
    {
      "Country": "Norway",
      "avg_age": 33.9022,
      "avg_entry_exam": 77.5102,
      "student_count": 49
    }
  ],
  "highest_avg_age": {
    "country": "Norway",
    "avg_age": 33.9022
  },
  "highest_avg_entry_exam": {
    "country": "Norway",
    "avg_entry_exam": 77.5102
  }
}


# Task 13

## Q

Compare entry exam scores between male and female students. Calculate the mean, median, and standard deviation for each gender. Which gender has a higher average entry exam score, and what is the absolute difference between the two averages?

## C

In [42]:
gender_stats = (df.groupby('Gender')['EntryExam']
               .agg(['mean', 'median', 'std'])
               .round(4)
               .reset_index())
gender_stats.columns = ['gender', 'mean', 'median', 'std']

means = df.groupby('Gender')['EntryExam'].mean()
higher = str(means.idxmax())
diff = float(abs(means.iloc[0] - means.iloc[1]))

result = {
    "answer_type": "comparison",
    "gender_stats": gender_stats.to_dict(orient='records'),
    "higher_gender": higher,
    "diff_absolute": round(diff, 4)
}
print(json_dumps_rounded(result, ensure_ascii=False, indent=2))

{
  "answer_type": "comparison",
  "gender_stats": [
    {
      "gender": "Female",
      "mean": 75.3256,
      "median": 80.0,
      "std": 16.5511
    },
    {
      "gender": "Male",
      "mean": 78.5588,
      "median": 81.5,
      "std": 16.4468
    }
  ],
  "higher_gender": "Male",
  "diff_absolute": 3.2332
}


# Task 14

## Q

Calculate the average Python course score and average database course score for each country. Which are the top 3 and bottom 3 countries by average Python score? Report all countries with both average scores, sorted by average Python score in descending order.

## C

In [43]:
country_scores = (df.groupby('Country')
                  .agg(avg_python=('Python', 'mean'),
                       avg_db=('Db', 'mean'))
                  .round(4)
                  .sort_values('avg_python', ascending=False)
                  .reset_index())

result = {
    "answer_type": "bidirectional_ranking",
    "primary_metric": "avg_python",
    "top3": country_scores.head(3).to_dict(orient='records'),
    "bottom3": country_scores.tail(3).to_dict(orient='records'),
    "all_countries": country_scores.to_dict(orient='records')
}
print(json_dumps_rounded(result, ensure_ascii=False, indent=2))

{
  "answer_type": "bidirectional_ranking",
  "primary_metric": "avg_python",
  "top3": [
    {
      "Country": "Germany",
      "avg_python": 87.0,
      "avg_db": 73.6667
    },
    {
      "Country": "United Kingdom",
      "avg_python": 85.0,
      "avg_db": 82.0
    },
    {
      "Country": "Denmark",
      "avg_python": 83.5,
      "avg_db": 86.5
    }
  ],
  "bottom3": [
    {
      "Country": "France",
      "avg_python": 72.0,
      "avg_db": 44.0
    },
    {
      "Country": "Italy",
      "avg_python": 69.0,
      "avg_db": 68.5
    },
    {
      "Country": "Nigeria",
      "avg_python": 51.0,
      "avg_db": 77.5
    }
  ],
  "all_countries": [
    {
      "Country": "Germany",
      "avg_python": 87.0,
      "avg_db": 73.6667
    },
    {
      "Country": "United Kingdom",
      "avg_python": 85.0,
      "avg_db": 82.0
    },
    {
      "Country": "Denmark",
      "avg_python": 83.5,
      "avg_db": 86.5
    },
    {
      "Country": "South Africa",
      "avg_python"

# Task 15

## Q

Among students belonging to the gender group with the higher average entry exam score, identify the top 6 countries by student count within that group. For each of these countries, report the number of students of that gender, their average Python score, and their average database score.

## C

In [44]:
# --- re-derive: which gender has higher avg EntryExam (from gender comparison) ---
gender_means = df.groupby('Gender')['EntryExam'].mean()
higher_gender = str(gender_means.idxmax())

# --- re-derive: country distribution (from country standardization) ---
gender_subset = df[df['Gender'] == higher_gender]

# --- novel computation: top countries within higher-scoring gender ---
top_countries = gender_subset['Country'].value_counts().head(6).index.tolist()

details = (gender_subset[gender_subset['Country'].isin(top_countries)]
           .groupby('Country')
           .agg(student_count=('Name', 'count'),
                avg_python=('Python', 'mean'),
                avg_db=('Db', 'mean'))
           .round(4)
           .sort_values('student_count', ascending=False)
           .reset_index())

result = {
    "answer_type": "ranking",
    "higher_gender": higher_gender,
    "top_countries_details": details.to_dict(orient='records')
}
print(json_dumps_rounded(result, ensure_ascii=False, indent=2))

{
  "answer_type": "ranking",
  "higher_gender": "Male",
  "top_countries_details": [
    {
      "Country": "Norway",
      "student_count": 20,
      "avg_python": 77.2427,
      "avg_db": 76.3
    },
    {
      "Country": "Kenya",
      "student_count": 3,
      "avg_python": 75.0,
      "avg_db": 70.0
    },
    {
      "Country": "Italy",
      "student_count": 2,
      "avg_python": 69.0,
      "avg_db": 68.5
    },
    {
      "Country": "Germany",
      "student_count": 2,
      "avg_python": 87.0,
      "avg_db": 80.0
    },
    {
      "Country": "Nigeria",
      "student_count": 2,
      "avg_python": 51.0,
      "avg_db": 77.5
    },
    {
      "Country": "Uganda",
      "student_count": 2,
      "avg_python": 77.5,
      "avg_db": 59.5
    }
  ]
}


# Task 16

## Q

Calculate the average database course score for each education level. Which education level has the highest average score and which has the lowest? Report all education levels with their average scores, sorted in descending order.

## C

In [45]:
edu_db = (df.groupby('PrevEducation')['Db']
          .mean()
          .round(4)
          .sort_values(ascending=False)
          .reset_index()
          .rename(columns={'PrevEducation': 'education_level', 'Db': 'avg_db_score'}))

result = {
    "answer_type": "ranking",
    "primary_metric": "avg_db_score",
    "highest": edu_db.iloc[0].to_dict(),
    "lowest": edu_db.iloc[-1].to_dict(),
    "all_levels": edu_db.to_dict(orient='records')
}
print(json_dumps_rounded(result, ensure_ascii=False, indent=2))

{
  "answer_type": "ranking",
  "primary_metric": "avg_db_score",
  "highest": {
    "education_level": "Masters",
    "avg_db_score": 77.0625
  },
  "lowest": {
    "education_level": "High School",
    "avg_db_score": 61.4211
  },
  "all_levels": [
    {
      "education_level": "Masters",
      "avg_db_score": 77.0625
    },
    {
      "education_level": "Bachelors",
      "avg_db_score": 70.88
    },
    {
      "education_level": "Doctorate",
      "avg_db_score": 69.8
    },
    {
      "education_level": "Diploma",
      "avg_db_score": 69.0
    },
    {
      "education_level": "High School",
      "avg_db_score": 61.4211
    }
  ]
}


# Task 17

## Q

Create a cross-tabulation of education level and gender showing the count of students in each combination. Which education level has the most balanced gender ratio, and which has the most imbalanced? Report the female percentage for each education level.

## C

In [46]:
ct = pd.crosstab(df['PrevEducation'], df['Gender'])
ct_records = ct.reset_index().to_dict(orient='records')

ct_analysis = ct.copy()
ct_analysis['total'] = ct_analysis.sum(axis=1)
ct_analysis['female_pct'] = (ct_analysis['Female'] / ct_analysis['total'] * 100).round(4)
ct_analysis['balance_score'] = (50 - (ct_analysis['female_pct'] - 50).abs()).round(4)

most_balanced = str(ct_analysis['balance_score'].idxmax())
most_imbalanced = str(ct_analysis['balance_score'].idxmin())

result = {
    "answer_type": "group_stat",
    "crosstab": ct_records,
    "most_balanced": {
        "education_level": most_balanced,
        "female_pct": float(ct_analysis.loc[most_balanced, 'female_pct'])
    },
    "most_imbalanced": {
        "education_level": most_imbalanced,
        "female_pct": float(ct_analysis.loc[most_imbalanced, 'female_pct'])
    }
}
print(json_dumps_rounded(result, ensure_ascii=False, indent=2))

{
  "answer_type": "group_stat",
  "crosstab": [
    {
      "PrevEducation": "Bachelors",
      "Female": 12,
      "Male": 13
    },
    {
      "PrevEducation": "Diploma",
      "Female": 4,
      "Male": 8
    },
    {
      "PrevEducation": "Doctorate",
      "Female": 2,
      "Male": 3
    },
    {
      "PrevEducation": "High School",
      "Female": 14,
      "Male": 5
    },
    {
      "PrevEducation": "Masters",
      "Female": 11,
      "Male": 5
    }
  ],
  "most_balanced": {
    "education_level": "Bachelors",
    "female_pct": 48.0
  },
  "most_imbalanced": {
    "education_level": "High School",
    "female_pct": 73.6842
  }
}


# Task 18

## Q

Among students whose age falls within the interquartile range and whose weekly study hours exceed the overall median, calculate the average entry exam score for each education level. How do these filtered averages compare to the unfiltered education-level averages? Report the difference for each education level and identify which level benefits most from this filtering.

## C

In [47]:
# --- re-derive: IQR bounds for Age (from outlier detection) ---
age_q1 = df['Age'].quantile(0.25)
age_q3 = df['Age'].quantile(0.75)

# --- re-derive: StudyHours median (from distribution analysis) ---
study_median = df['StudyHours'].median()

# --- re-derive: unfiltered avg EntryExam by education (from education analysis) ---
unfiltered_avg = df.groupby('PrevEducation')['EntryExam'].mean()

# --- novel computation: multi-condition filtered comparison ---
filtered = df[(df['Age'] >= age_q1) & (df['Age'] <= age_q3) & (df['StudyHours'] > study_median)]
filtered_avg = filtered.groupby('PrevEducation')['EntryExam'].mean()
filtered_count = filtered.groupby('PrevEducation').size()

comparison = pd.DataFrame({
    'unfiltered_avg': unfiltered_avg,
    'filtered_avg': filtered_avg,
    'filtered_count': filtered_count
}).reindex(unfiltered_avg.index)
comparison['filtered_count'] = comparison['filtered_count'].fillna(0).astype(int)
comparison['difference'] = comparison['filtered_avg'] - comparison['unfiltered_avg']

comparison = comparison.round(4).reset_index()
comparison = comparison.rename(columns={'PrevEducation': 'education_level'})
comparison = comparison.sort_values(
    ['difference', 'education_level'],
    ascending=[False, True],
    na_position='last'
).reset_index(drop=True)

benefit_candidates = comparison.dropna(subset=['difference'])
most_benefit = None if benefit_candidates.empty else benefit_candidates.iloc[0].to_dict()

result = {
    "answer_type": "comparison",
    "filter_criteria": {
        "age_q1": round(float(age_q1), 4),
        "age_q3": round(float(age_q3), 4),
        "study_hours_median": round(float(study_median), 4)
    },
    "total_filtered_students": int(len(filtered)),
    "comparison": json.loads(comparison.to_json(orient='records')),
    "most_benefit": None if most_benefit is None else json.loads(pd.Series(most_benefit).to_json())
}
print(json_dumps_rounded(result, ensure_ascii=False, indent=2))


{
  "answer_type": "comparison",
  "filter_criteria": {
    "age_q1": 27.0,
    "age_q3": 41.0,
    "study_hours_median": 156.0
  },
  "total_filtered_students": 16,
  "comparison": [
    {
      "education_level": "Diploma",
      "unfiltered_avg": 70.1667,
      "filtered_avg": 95.0,
      "filtered_count": 1,
      "difference": 24.8333
    },
    {
      "education_level": "Doctorate",
      "unfiltered_avg": 79.6,
      "filtered_avg": 94.0,
      "filtered_count": 1,
      "difference": 14.4
    },
    {
      "education_level": "Bachelors",
      "unfiltered_avg": 80.64,
      "filtered_avg": 87.0,
      "filtered_count": 5,
      "difference": 6.36
    },
    {
      "education_level": "Masters",
      "unfiltered_avg": 85.1875,
      "filtered_avg": 89.5556,
      "filtered_count": 9,
      "difference": 4.3681
    },
    {
      "education_level": "High School",
      "unfiltered_avg": 67.9474,
      "filtered_avg": null,
      "filtered_count": 0,
      "difference": null
  

# Task 19

## Context

We now move to multivariate analysis to understand the relationships between all numeric features. A Pearson correlation matrix reveals the linear associations between age, entry exam score, study hours, Python score, and database score.

## Q

Compute the Pearson correlation matrix for all five numeric features. What are the 3 strongest correlated pairs and the 3 weakest correlated pairs by absolute value? Report the full matrix and the ranked pairs.

## C

In [48]:
numeric_cols = ['Age', 'EntryExam', 'StudyHours', 'Python', 'Db']
corr_matrix = df[numeric_cols].corr().round(4)

pairs = []
for i in range(len(numeric_cols)):
    for j in range(i + 1, len(numeric_cols)):
        pairs.append({
            "col_a": numeric_cols[i],
            "col_b": numeric_cols[j],
            "correlation": round(float(corr_matrix.iloc[i, j]), 4)
        })

pairs_desc = sorted(pairs, key=lambda x: abs(x['correlation']), reverse=True)
pairs_asc = sorted(pairs, key=lambda x: abs(x['correlation']))

result = {
    "answer_type": "correlation_matrix",
    "matrix": {col: {c: round(float(corr_matrix.loc[col, c]), 4)
                     for c in numeric_cols} for col in numeric_cols},
    "top3_strongest": pairs_desc[:3],
    "top3_weakest": pairs_asc[:3]
}
print(json_dumps_rounded(result, ensure_ascii=False, indent=2))


{
  "answer_type": "correlation_matrix",
  "matrix": {
    "Age": {
      "Age": 1.0,
      "EntryExam": 0.2588,
      "StudyHours": 0.3435,
      "Python": 0.1062,
      "Db": 0.1514
    },
    "EntryExam": {
      "Age": 0.2588,
      "EntryExam": 1.0,
      "StudyHours": 0.8079,
      "Python": 0.7593,
      "Db": 0.6085
    },
    "StudyHours": {
      "Age": 0.3435,
      "EntryExam": 0.8079,
      "StudyHours": 1.0,
      "Python": 0.7837,
      "Db": 0.466
    },
    "Python": {
      "Age": 0.1062,
      "EntryExam": 0.7593,
      "StudyHours": 0.7837,
      "Python": 1.0,
      "Db": 0.4427
    },
    "Db": {
      "Age": 0.1514,
      "EntryExam": 0.6085,
      "StudyHours": 0.466,
      "Python": 0.4427,
      "Db": 1.0
    }
  },
  "top3_strongest": [
    {
      "col_a": "EntryExam",
      "col_b": "StudyHours",
      "correlation": 0.8079
    },
    {
      "col_a": "StudyHours",
      "col_b": "Python",
      "correlation": 0.7837
    },
    {
      "col_a": "EntryExam",

# Task 20

## Q

For countries with a student count above the median country size, compute the Pearson correlation between Python and database scores within that subset. Compare this to the full-dataset correlation between the same two variables. Is the relationship stronger or weaker among students from larger-population countries? Report the qualifying countries and both correlation values.

## C

In [49]:
# --- re-derive: country student counts (from country standardization) ---
country_counts = df['Country'].value_counts()
median_count = float(country_counts.median())
large_countries = country_counts[country_counts > median_count].index.tolist()

# --- re-derive: full Python-Db correlation (from correlation matrix) ---
full_corr = float(df['Python'].corr(df['Db']))

# --- novel computation: subset correlation for large countries ---
subset = df[df['Country'].isin(large_countries)]
subset_corr = float(subset['Python'].corr(subset['Db']))

result = {
    "answer_type": "correlation",
    "median_country_size": round(median_count, 4),
    "large_countries": large_countries,
    "num_students_in_subset": int(len(subset)),
    "full_dataset_correlation": round(full_corr, 4),
    "subset_correlation": round(subset_corr, 4),
    "difference": round(subset_corr - full_corr, 4),
    "stronger_in_subset": bool(abs(subset_corr) > abs(full_corr))
}
print(json_dumps_rounded(result, ensure_ascii=False, indent=2))

{
  "answer_type": "correlation",
  "median_country_size": 2.0,
  "large_countries": [
    "Norway",
    "Uganda",
    "Kenya",
    "Germany"
  ],
  "num_students_in_subset": 59,
  "full_dataset_correlation": 0.4427,
  "subset_correlation": 0.5108,
  "difference": 0.0682,
  "stronger_in_subset": true
}


# Task 21

## Q

Using the linear relationship between study hours and entry exam scores, predict the expected entry exam score for the average study hours of each education level. Compare these predicted scores to the actual average entry exam scores per education level. Which education level outperforms its study-hours-based prediction the most (largest positive residual)? Report the regression slope, intercept, and the full comparison table.

## C

In [50]:
# --- re-derive: StudyHours-EntryExam relationship (from correlation analysis) ---
valid = df[['StudyHours', 'EntryExam']].dropna()
coeffs = np.polyfit(valid['StudyHours'].values, valid['EntryExam'].values, 1)

# --- re-derive: education-level grouping (from education analysis) ---
edu_actual = df.groupby('PrevEducation').agg(
    avg_study_hours=('StudyHours', 'mean'),
    actual_avg_exam=('EntryExam', 'mean')
).reset_index()

# --- novel computation: predicted vs actual by education level ---
edu_actual['predicted_exam'] = np.poly1d(coeffs)(edu_actual['avg_study_hours'])
edu_actual['residual'] = edu_actual['actual_avg_exam'] - edu_actual['predicted_exam']
edu_actual = edu_actual.sort_values('residual', ascending=False).round(4)

best = edu_actual.iloc[0]

result = {
    "answer_type": "regression",
    "model": "linear (StudyHours -> EntryExam)",
    "slope": round(float(coeffs[0]), 4),
    "intercept": round(float(coeffs[1]), 4),
    "education_comparison": edu_actual.rename(
        columns={'PrevEducation': 'education_level'}).to_dict(orient='records'),
    "largest_positive_residual": {
        "education_level": str(best['PrevEducation']),
        "residual": round(float(best['residual']), 4)
    }
}
print(json_dumps_rounded(result, ensure_ascii=False, indent=2))

{
  "answer_type": "regression",
  "model": "linear (StudyHours -> EntryExam)",
  "slope": 1.0445,
  "intercept": -79.6191,
  "education_comparison": [
    {
      "education_level": "High School",
      "avg_study_hours": 138.1053,
      "actual_avg_exam": 67.9474,
      "predicted_exam": 64.628,
      "residual": 3.3194
    },
    {
      "education_level": "Masters",
      "avg_study_hours": 156.75,
      "actual_avg_exam": 85.1875,
      "predicted_exam": 84.1019,
      "residual": 1.0856
    },
    {
      "education_level": "Doctorate",
      "avg_study_hours": 151.6,
      "actual_avg_exam": 79.6,
      "predicted_exam": 78.7228,
      "residual": 0.8772
    },
    {
      "education_level": "Bachelors",
      "avg_study_hours": 154.64,
      "actual_avg_exam": 80.64,
      "predicted_exam": 81.898,
      "residual": -1.258
    },
    {
      "education_level": "Diploma",
      "avg_study_hours": 147.6667,
      "actual_avg_exam": 70.1667,
      "predicted_exam": 74.6146,
      

# Task 22

## Q

For each residence type, calculate the average entry exam score separately for male and female students, then compute the gender gap (male average minus female average). Which residence type shows the largest absolute gender gap in entry exam performance? Report the full breakdown for all residence types.

## C

In [51]:
# --- re-derive: residence types (from residence standardization) ---
# --- re-derive: gender-based comparison approach (from gender analysis) ---

# --- novel computation: gender gap by residence ---
pivot = (df.groupby(['Residence', 'Gender'])['EntryExam']
         .mean()
         .unstack(fill_value=0)
         .round(4))
pivot['gender_gap'] = (pivot['Male'] - pivot['Female']).round(4)
pivot['abs_gap'] = pivot['gender_gap'].abs()
pivot = pivot.sort_values('abs_gap', ascending=False).reset_index()

largest = pivot.iloc[0]

result = {
    "answer_type": "comparison",
    "residence_gender_gaps": pivot[['Residence', 'Female', 'Male', 'gender_gap']].to_dict(orient='records'),
    "largest_absolute_gap": {
        "residence": str(largest['Residence']),
        "male_avg": round(float(largest['Male']), 4),
        "female_avg": round(float(largest['Female']), 4),
        "gap": round(float(largest['gender_gap']), 4)
    }
}
print(json_dumps_rounded(result, ensure_ascii=False, indent=2))

{
  "answer_type": "comparison",
  "residence_gender_gaps": [
    {
      "Residence": "BI Residence",
      "Female": 72.8333,
      "Male": 80.0,
      "gender_gap": 7.1667
    },
    {
      "Residence": "Sognsvann",
      "Female": 71.8,
      "Male": 76.5714,
      "gender_gap": 4.7714
    },
    {
      "Residence": "Private",
      "Female": 78.45,
      "Male": 78.0769,
      "gender_gap": -0.3731
    }
  ],
  "largest_absolute_gap": {
    "residence": "BI Residence",
    "male_avg": 80.0,
    "female_avg": 72.8333,
    "gap": 7.1667
  }
}


# Task 23

## Q

Define "academically strong" countries as those where both the average entry exam score and the average Python score exceed their respective overall country-level medians. How many countries qualify as academically strong? List them in country-name ascending order with their average entry exam, Python, and database scores. What is the total number of students from these countries, and what is their gender distribution?

## C

In [52]:
# --- re-derive: country-level averages (from country score ranking) ---
country_avgs = df.groupby('Country').agg(
    avg_exam=('EntryExam', 'mean'),
    avg_python=('Python', 'mean'),
    avg_db=('Db', 'mean')
)

# --- re-derive: gender analysis approach (from gender comparison) ---

# --- novel computation: multi-threshold country classification ---
median_exam = float(country_avgs['avg_exam'].median())
median_python = float(country_avgs['avg_python'].median())

strong = country_avgs[
    (country_avgs['avg_exam'] > median_exam) &
    (country_avgs['avg_python'] > median_python)
].round(4).reset_index()

strong_countries = strong['Country'].tolist()
students_in_strong = df[df['Country'].isin(strong_countries)]
gender_split = students_in_strong['Gender'].value_counts().to_dict()

result = {
    "answer_type": "ranking",
    "thresholds": {
        "median_exam": round(median_exam, 4),
        "median_python": round(median_python, 4)
    },
    "num_qualifying_countries": len(strong),
    "qualifying_countries": strong.to_dict(orient='records'),
    "total_students": int(len(students_in_strong)),
    "gender_split": {str(k): int(v) for k, v in gender_split.items()}
}
print(json_dumps_rounded(result, ensure_ascii=False, indent=2))

{
  "answer_type": "ranking",
  "thresholds": {
    "median_exam": 77.0,
    "median_python": 78.0
  },
  "num_qualifying_countries": 4,
  "qualifying_countries": [
    {
      "Country": "Denmark",
      "avg_exam": 87.5,
      "avg_python": 83.5,
      "avg_db": 86.5
    },
    {
      "Country": "Germany",
      "avg_exam": 92.3333,
      "avg_python": 87.0,
      "avg_db": 73.6667
    },
    {
      "Country": "Spain",
      "avg_exam": 79.0,
      "avg_python": 79.0,
      "avg_db": 58.5
    },
    {
      "Country": "United Kingdom",
      "avg_exam": 93.0,
      "avg_python": 85.0,
      "avg_db": 82.0
    }
  ],
  "total_students": 9,
  "gender_split": {
    "Female": 5,
    "Male": 4
  }
}


# Task 24

## Context

For predictive modeling, we use the cleaned dataset to predict entry exam scores. All categorical features (excluding the student name) are label-encoded to convert them to numeric form. The data is split into 80% training and 20% test sets using random_state=42. Five regression models are trained and compared: Linear Regression, Random Forest, Gradient Boosting, SVR, and XGBoost.

## Q

Train the five regression models described above to predict entry exam scores. Report each model's R-squared percentage, MAE, and RMSE on the test set in the same order the models are listed in the context. Which model achieves the best R-squared?

## C

In [53]:
df_model = df.copy()
cat_cols_model = df_model.select_dtypes(exclude=['number']).columns.tolist()
num_cols_model = df_model.select_dtypes(include=['number']).columns.tolist()

df_model[num_cols_model] = df_model[num_cols_model].fillna(df_model[num_cols_model].median())
for col in cat_cols_model:
    df_model[col] = df_model[col].fillna(df_model[col].mode()[0])

le = LabelEncoder()
for col in cat_cols_model:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

X = df_model.drop(columns=['Name', 'EntryExam'])
y = df_model['EntryExam']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "SVR": SVR(),
    "XGBoost": XGBRegressor(random_state=42)
}

results_list = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2 = float(r2_score(y_test, y_pred) * 100)
    mae = float(mean_absolute_error(y_test, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
    results_list.append({
        "model": name,
        "r2_pct": round(r2, 4),
        "mae": round(mae, 4),
        "rmse": round(rmse, 4)
    })

best = max(results_list, key=lambda x: x['r2_pct'])

result = {
    "answer_type": "ranking",
    "primary_metric": "r2_pct",
    "train_size": int(len(X_train)),
    "test_size": int(len(X_test)),
    "num_features": int(X.shape[1]),
    "feature_names": X.columns.tolist(),
    "model_results": results_list,
    "best_model": best
}
print(json_dumps_rounded(result, ensure_ascii=False, indent=2))

{
  "answer_type": "ranking",
  "primary_metric": "r2_pct",
  "train_size": 61,
  "test_size": 16,
  "num_features": 8,
  "feature_names": [
    "Age",
    "Gender",
    "Country",
    "Residence",
    "PrevEducation",
    "StudyHours",
    "Python",
    "Db"
  ],
  "model_results": [
    {
      "model": "Linear Regression",
      "r2_pct": 75.7132,
      "mae": 6.8795,
      "rmse": 8.6286
    },
    {
      "model": "Random Forest",
      "r2_pct": 61.5472,
      "mae": 7.7988,
      "rmse": 10.8573
    },
    {
      "model": "Gradient Boosting",
      "r2_pct": 54.3376,
      "mae": 7.8749,
      "rmse": 11.8314
    },
    {
      "model": "SVR",
      "r2_pct": 9.0084,
      "mae": 12.8794,
      "rmse": 16.7016
    },
    {
      "model": "XGBoost",
      "r2_pct": 57.8853,
      "mae": 7.8021,
      "rmse": 11.3625
    }
  ],
  "best_model": {
    "model": "Linear Regression",
    "r2_pct": 75.7132,
    "mae": 6.8795,
    "rmse": 8.6286
  }
}


# Task 25

## Q

Select only the top 3 features most strongly correlated with entry exam scores (by absolute Pearson correlation from the encoded feature set) and retrain the best-performing model using just those features. Compare the R-squared, MAE, and RMSE to the full-feature model. Does reducing to the top 3 correlated features significantly impact predictive performance?

## C

In [54]:
# --- re-derive: feature correlations with EntryExam (from correlation analysis) ---
feature_cols = X.columns.tolist()
exam_corrs = {col: abs(float(df_model[col].corr(df_model['EntryExam'])))
              for col in feature_cols}
top3_features = sorted(exam_corrs, key=exam_corrs.get, reverse=True)[:3]

# --- re-derive: best model identity (from model comparison) ---
model_r2 = {}
for name, model in models.items():
    y_pred = model.predict(X_test)
    model_r2[name] = float(r2_score(y_test, y_pred))
best_model_name = max(model_r2, key=model_r2.get)

# --- novel computation: retrain with top 3 features ---
X_train_top3 = X_train[top3_features]
X_test_top3 = X_test[top3_features]

model_class = models[best_model_name].__class__
if best_model_name in ["Random Forest", "Gradient Boosting", "XGBoost"]:
    reduced_model = model_class(random_state=42)
else:
    reduced_model = model_class()

reduced_model.fit(X_train_top3, y_train)
y_pred_reduced = reduced_model.predict(X_test_top3)

full_r2 = round(model_r2[best_model_name] * 100, 4)
reduced_r2 = round(float(r2_score(y_test, y_pred_reduced)) * 100, 4)
reduced_mae = round(float(mean_absolute_error(y_test, y_pred_reduced)), 4)
reduced_rmse = round(float(np.sqrt(mean_squared_error(y_test, y_pred_reduced))), 4)

result = {
    "answer_type": "comparison",
    "best_model": best_model_name,
    "top3_features": [{"feature": f, "abs_correlation": round(exam_corrs[f], 4)}
                      for f in top3_features],
    "full_model_r2_pct": full_r2,
    "reduced_model_r2_pct": reduced_r2,
    "reduced_model_mae": reduced_mae,
    "reduced_model_rmse": reduced_rmse,
    "r2_change_pct": round(reduced_r2 - full_r2, 4),
    "significant_impact": abs(reduced_r2 - full_r2) > 5
}
print(json_dumps_rounded(result, ensure_ascii=False, indent=2))

{
  "answer_type": "comparison",
  "best_model": "Linear Regression",
  "top3_features": [
    {
      "feature": "StudyHours",
      "abs_correlation": 0.8079
    },
    {
      "feature": "Python",
      "abs_correlation": 0.7593
    },
    {
      "feature": "Db",
      "abs_correlation": 0.6085
    }
  ],
  "full_model_r2_pct": 75.7132,
  "reduced_model_r2_pct": 81.5432,
  "reduced_model_mae": 5.8336,
  "reduced_model_rmse": 7.522,
  "r2_change_pct": 5.83,
  "significant_impact": true
}


# Task 26

## Q

Compute a composite academic z-score for each student by standardizing (zero mean, unit variance) the entry exam, Python, and database scores, then averaging the three z-scores per student. Who are the top 5 students by this composite score? Report their names, countries, education levels, individual z-scores, and composite scores.

## C

In [55]:
# --- re-derive: score distributions (from descriptive statistics) ---
scaler = StandardScaler()
z_scores = pd.DataFrame(
    scaler.fit_transform(df[['EntryExam', 'Python', 'Db']]),
    columns=['z_EntryExam', 'z_Python', 'z_Db'],
    index=df.index
)

# --- novel computation: composite z-score ranking ---
z_scores['composite'] = z_scores[['z_EntryExam', 'z_Python', 'z_Db']].mean(axis=1)
df_ranked = pd.concat([df[['Name', 'Country', 'PrevEducation']], z_scores], axis=1)
top5 = (df_ranked.sort_values('composite', ascending=False)
        .head(5)
        [['Name', 'Country', 'PrevEducation', 'z_EntryExam', 'z_Python', 'z_Db', 'composite']]
        .round(4))

result = {
    "answer_type": "ranking",
    "primary_metric": "composite_z_score",
    "top5": top5.to_dict(orient='records')
}
print(json_dumps_rounded(result, ensure_ascii=False, indent=2))

{
  "answer_type": "ranking",
  "primary_metric": "composite_z_score",
  "top5": [
    {
      "Name": "Sindre Hansen",
      "Country": "Norway",
      "PrevEducation": "Bachelors",
      "z_EntryExam": 1.1148,
      "z_Python": 0.7378,
      "z_Db": 1.7451,
      "composite": 1.1992
    },
    {
      "Name": "Line Næss",
      "Country": "Norway",
      "PrevEducation": "High School",
      "z_EntryExam": 1.298,
      "z_Python": 0.9364,
      "z_Db": 1.2133,
      "composite": 1.1493
    },
    {
      "Name": "Grethe Brekke",
      "Country": "Norway",
      "PrevEducation": "Masters",
      "z_EntryExam": 1.1758,
      "z_Python": 0.7378,
      "z_Db": 1.3315,
      "composite": 1.0817
    },
    {
      "Name": "Odd Knudsen",
      "Country": "Norway",
      "PrevEducation": "Diploma",
      "z_EntryExam": 0.8704,
      "z_Python": 0.6055,
      "z_Db": 1.7451,
      "composite": 1.0737
    },
    {
      "Name": "Dag Arnesen",
      "Country": "Norway",
      "PrevEducation": "

# Task 27

## Q

Compare the feature importance rankings from the best-performing tree-based model with the univariate absolute Pearson correlation rankings between each feature and entry exam scores. For each feature, report its model-based importance score and its absolute correlation with entry exam scores. Do the two ranking methods agree on the most important feature? Calculate Spearman's rank correlation between the two rankings to quantify agreement.

## C

In [56]:
# --- re-derive: Pearson correlations with EntryExam (from correlation analysis) ---
feature_cols = X.columns.tolist()
pearson_corrs = {col: round(abs(float(df_model[col].corr(df_model['EntryExam']))), 4)
                 for col in feature_cols}

# --- re-derive: best tree-based model (from model comparison) ---
tree_models = {name: model for name, model in models.items()
               if hasattr(model, 'feature_importances_')}
best_tree_name = max(tree_models,
                     key=lambda n: float(r2_score(y_test, tree_models[n].predict(X_test))))
best_tree = tree_models[best_tree_name]

# --- novel computation: importance vs correlation comparison ---
importances = dict(zip(feature_cols,
                       [round(float(v), 4) for v in best_tree.feature_importances_]))

comp_df = pd.DataFrame({
    'feature': feature_cols,
    'model_importance': [importances[f] for f in feature_cols],
    'abs_pearson_corr': [pearson_corrs[f] for f in feature_cols]
})
comp_df['importance_rank'] = comp_df['model_importance'].rank(ascending=False).astype(int)
comp_df['correlation_rank'] = comp_df['abs_pearson_corr'].rank(ascending=False).astype(int)
comp_df = comp_df.sort_values('model_importance', ascending=False)

spearman_val, p_val = spearmanr(
    comp_df['importance_rank'], comp_df['correlation_rank'])

top_imp_feature = comp_df.iloc[0]['feature']
top_corr_feature = comp_df.sort_values('abs_pearson_corr', ascending=False).iloc[0]['feature']

result = {
    "answer_type": "comparison",
    "best_tree_model": best_tree_name,
    "feature_comparison": comp_df.round(4).to_dict(orient='records'),
    "top_by_importance": str(top_imp_feature),
    "top_by_correlation": str(top_corr_feature),
    "rankings_agree_on_top": str(top_imp_feature) == str(top_corr_feature),
    "spearman_rank_correlation": round(float(spearman_val), 4),
    "spearman_p_value": round(float(p_val), 4)
}
print(json_dumps_rounded(result, ensure_ascii=False, indent=2))

{
  "answer_type": "comparison",
  "best_tree_model": "Random Forest",
  "feature_comparison": [
    {
      "feature": "StudyHours",
      "model_importance": 0.6739,
      "abs_pearson_corr": 0.8079,
      "importance_rank": 1,
      "correlation_rank": 1
    },
    {
      "feature": "Db",
      "model_importance": 0.1773,
      "abs_pearson_corr": 0.6085,
      "importance_rank": 2,
      "correlation_rank": 3
    },
    {
      "feature": "Python",
      "model_importance": 0.0611,
      "abs_pearson_corr": 0.7593,
      "importance_rank": 3,
      "correlation_rank": 2
    },
    {
      "feature": "Residence",
      "model_importance": 0.0247,
      "abs_pearson_corr": 0.001,
      "importance_rank": 4,
      "correlation_rank": 8
    },
    {
      "feature": "Age",
      "model_importance": 0.0219,
      "abs_pearson_corr": 0.2588,
      "importance_rank": 5,
      "correlation_rank": 4
    },
    {
      "feature": "Country",
      "model_importance": 0.0204,
      "abs_pears